# Phase 1: Data Cleaning & Quality Check
**Project:** IndiaKart E-Commerce Analytics
**Name:** MADDULA KEERTHI PRIYANKA
**Domain:** Data Analyst (Internmo Virtual Internship)
**Data period:** 24-Jun-2023 to 23-Jun-2025

In [2]:
%pip install pandas numpy matplotlib seaborn openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
## Importing Required Libraries

In [4]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [5]:
path = "data/"

customers   = pd.read_csv(path + "customers DA.csv")
inventory   = pd.read_csv(path + "inventory DA.csv")
order_items = pd.read_csv(path + "order_items DA.csv")
orders      = pd.read_csv(path + "orders DA.csv")
payments    = pd.read_csv(path + "payments DA.csv")
products    = pd.read_csv(path + "products DA.csv")
returns     = pd.read_csv(path + "returns DA.csv")
suppliers   = pd.read_csv(path + "suppliers DA.csv")

tables = {
    "customers": customers, "inventory": inventory,
    "order_items": order_items, "orders": orders,
    "payments": payments, "products": products,
    "returns": returns, "suppliers": suppliers,
}

In [6]:
profile = pd.DataFrame({
    "table": list(tables.keys()),
    "rows": [df.shape[0] for df in tables.values()],
    "columns": [df.shape[1] for df in tables.values()],
})
profile

,table,rows,columns
0,customers,10000,17
1,inventory,1000,11
2,order_items,100000,11
3,orders,50000,19
4,payments,50000,13
5,products,1000,17
6,returns,10000,10
7,suppliers,200,14


## 3. Missing Values

In [7]:
null_report = []
for name, df in tables.items():
    for col in df.columns:
        n = df[col].isna().sum()
        if n > 0:
            null_report.append({
                "table": name,
                "column": col,
                "null_count": n,
                "null_pct": round(n / len(df) * 100, 2),
            })

null_df = pd.DataFrame(null_report)
null_df if not null_df.empty else "No missing values found"

,table,column,null_count,null_pct
0,orders,delivered_date,17501,35.00
1,payments,refund_date,50000,100.00


## 4. Duplicate Checks

In [8]:
# Fully duplicated rows in each table
dup_rows = {name: df.duplicated().sum() for name, df in tables.items()}
print("Duplicate rows per table:")
print(dup_rows)

# Duplicate primary keys
keys = {
    "customers": "customer_id", "inventory": "inventory_id",
    "order_items": "item_id", "orders": "order_id",
    "payments": "payment_id", "products": "product_id",
    "returns": "return_id", "suppliers": "supplier_id",
}
print("\nDuplicate primary keys per table:")
for name, key in keys.items():
    print(f"{name:12s} {key:14s} {tables[name][key].duplicated().sum()}")

Duplicate rows per table:
{'customers': np.int64(0), 'inventory': np.int64(0), 'order_items': np.int64(0), 'orders': np.int64(0), 'payments': np.int64(0), 'products': np.int64(0), 'returns': np.int64(0), 'suppliers': np.int64(0)}

Duplicate primary keys per table:
customers    customer_id    0
inventory    inventory_id   0
order_items  item_id        0
orders       order_id       0
payments     payment_id     0
products     product_id     0
returns      return_id      0
suppliers    supplier_id    0


## 5. Data Type Validation

In [9]:
orders.dtypes

order_id                str
customer_id             str
order_date              str
order_time              str
status                  str
city                    str
state                   str
pincode               int64
total_amount        float64
gst_amount          float64
shipping_charge       int64
discount_amount     float64
final_amount        float64
payment_method          str
shipping_partner        str
tracking_id             str
delivered_date          str
is_cod                int64
channel                 str
dtype: object

In [10]:
date_cols = {
    "orders": ["order_date", "delivered_date"],
    "customers": ["registration_date", "last_login_date"],
    "payments": ["payment_date", "refund_date"],
    "returns": ["return_date", "refund_date"],
    "products": ["launch_date"],
    "inventory": ["last_restocked_date"],
    "suppliers": ["created_date"],
}

for t, cols in date_cols.items():
    for c in cols:
        print(t, c, "->", tables[t][c].dtype)

orders order_date -> str
orders delivered_date -> str
customers registration_date -> str
customers last_login_date -> str
payments payment_date -> str
payments refund_date -> float64
returns return_date -> str
returns refund_date -> str
products launch_date -> str
inventory last_restocked_date -> str
suppliers created_date -> str


In [11]:
for t, cols in date_cols.items():
    for c in cols:
        tables[t][c] = pd.to_datetime(tables[t][c], format="%d-%m-%Y", errors="coerce")

orders[["order_date", "delivered_date"]].dtypes

order_date        datetime64[us]
delivered_date    datetime64[us]
dtype: object

## 6. Outlier Detection

In [12]:
print(orders["final_amount"].describe())

high_value = orders[orders["final_amount"] > 500000]
print("\nOrders above Rs. 5 lakh:", len(high_value))
high_value[["order_id", "customer_id", "status", "final_amount"]].head(10)

count      50,000.00
mean       62,947.71
std        97,897.50
min           136.07
25%         6,893.52
50%        24,183.38
75%        71,495.74
max     1,133,382.48
Name: final_amount, dtype: float64

Orders above Rs. 5 lakh: 312


,order_id,customer_id,status,final_amount
33,ORD000034,CUST01867,Delivered,"527,178.05"
34,ORD000035,CUST01370,Cancelled,"532,061.13"
306,ORD000307,CUST03874,Cancelled,"550,716.17"
583,ORD000584,CUST07624,Delivered,"745,667.68"
590,ORD000591,CUST05298,Cancelled,"520,293.72"
763,ORD000764,CUST00862,Delivered,"540,156.71"
985,ORD000986,CUST06894,Shipped,"512,079.51"
1092,ORD001093,CUST00492,Cancelled,"680,009.23"
1114,ORD001115,CUST03544,Delivered,"646,223.50"
1178,ORD001179,CUST07815,Processing,"513,624.71"


In [13]:
print("Negative final_amount:", (orders["final_amount"] < 0).sum())
print("Zero/negative quantity:", (order_items["quantity"] <= 0).sum())
print("Age outside 15-100:", ((customers["age"] < 15) | (customers["age"] > 100)).sum())

Negative final_amount: 0
Zero/negative quantity: 0
Age outside 15-100: 0


## 7. Referential Integrity

In [14]:
checks = [
    ("order_items.order_id -> orders",    order_items["order_id"],   orders["order_id"]),
    ("order_items.product_id -> products", order_items["product_id"], products["product_id"]),
    ("orders.customer_id -> customers",   orders["customer_id"],     customers["customer_id"]),
    ("payments.order_id -> orders",       payments["order_id"],      orders["order_id"]),
    ("returns.order_id -> orders",        returns["order_id"],       orders["order_id"]),
    ("products.supplier_id -> suppliers", products["supplier_id"],   suppliers["supplier_id"]),
    ("inventory.product_id -> products",  inventory["product_id"],   products["product_id"]),
]

for label, child, parent in checks:
    orphans = (~child.isin(parent)).sum()
    print(f"{label:38s} orphans: {orphans}")

order_items.order_id -> orders         orphans: 0
order_items.product_id -> products     orphans: 0
orders.customer_id -> customers        orphans: 0
payments.order_id -> orders            orphans: 0
returns.order_id -> orders             orphans: 0
products.supplier_id -> suppliers      orphans: 0
inventory.product_id -> products       orphans: 0


## 8. Specific Business Checks

In [15]:
# 1. Delivery date must not be before order date
bad_delivery = (orders["delivered_date"] < orders["order_date"]).sum()
print("1. Delivered before ordered:", bad_delivery)

# 2. Payment failure rate
fail_rate = (payments["status"] == "Failed").mean() * 100
print(f"2. Payment failure rate: {fail_rate:.2f}%")

# 3. Returns with no matching order
print("3. Returns with no matching order:", (~returns["order_id"].isin(orders["order_id"])).sum())

# 4. Out of Stock products
print("4. Out of Stock products:", (inventory["status"] == "Out of Stock").sum())

# 5. Customers with zero orders
print("5. Customers with total_orders = 0:", (customers["total_orders"] == 0).sum())

1. Delivered before ordered: 0
2. Payment failure rate: 3.50%
3. Returns with no matching order: 0
4. Out of Stock products: 2
5. Customers with total_orders = 0: 457


## 9. Data Quality Report

**Overview:** 8 tables with 222,200 rows in total, covering 24-Jun-2023 to 23-Jun-2025.

| Check | Result | Status |
|---|---|---|
| Duplicate rows / primary keys | 0 in all 8 tables | Pass |
| Orphan foreign keys (7 relationships) | 0 | Pass |
| Delivered before ordered | 0 | Pass |
| Returns with no matching order | 0 | Pass |
| Negative amounts / invalid quantity / invalid age | 0 | Pass |
| Missing values | 2 columns | Noted |
| Payment failure rate | 3.50% (target below 2%) | Flag |
| Orders above Rs. 5 lakh | 312 (0.62%) | Flag |
| Out of Stock products | 2 | Noted |
| Customers with 0 orders | 457 (4.57%) | Noted |

**Issues found**
- delivered_date is null for 17,501 orders (35%); payments.refund_date is 100% null.
- final_amount is right-skewed (mean Rs. 62,948 vs median Rs. 24,183).
- returns.csv has 10,000 rows, but only 3,933 orders have status "Returned".

**Actions taken**
- Converted all date columns from text to datetime (DD-MM-YYYY).
- Kept nulls in delivered_date: they are legitimate for orders not delivered.
- Kept refund_date; it can't support refund timing analysis.
- No rows removed (no duplicates or invalid values).
- Flagged high-value orders but did not remove them.

**Notes for later phases**
- Filter by order status before calculating revenue.
- Define return rate clearly (returns table vs. Returned status) and state the definition used.

In [16]:
import os
os.makedirs("cleaned", exist_ok=True)

for name, df in tables.items():
    df.to_csv(f"cleaned/{name}.csv", index=False)

print("Saved:", os.listdir("cleaned"))

Saved: ['customers.csv', 'inventory.csv', 'orders.csv', 'order_items.csv', 'payments.csv', 'products.csv', 'returns.csv', 'suppliers.csv']
